In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_squared_error
)

In [ ]:
# Convert images into tensors and normalize them
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Training dataset
train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Testing dataset
test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 6))

for i in range(10):
    plt.subplot(2, 5, i + 1)

    image = images[i].squeeze()

    plt.imshow(image, cmap="gray")
    plt.title(class_names[labels[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU()

        self.fc1 = nn.Linear(64 * 7 * 7, 128)

        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.relu(self.conv1(x))

        x = self.pool(x)

        x = self.relu(self.conv2(x))

        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))

        x = self.fc2(x)

        return x

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = CNN().to(device)

print(model)
print("Device:", device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 10

train_losses = []
train_accuracies = []

In [ ]:
for epoch in range(epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

        # Find predicted class
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)

    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)

    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, epochs + 1),
    train_losses,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss vs Epoch")

plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, epochs + 1),
    train_accuracies,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Accuracy (%)")
plt.title("Training Accuracy vs Epoch")

plt.grid(True)
plt.show()

In [ ]:
model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

test_loss = 0.0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        test_loss += loss.item()

        probabilities = torch.softmax(outputs, dim=1)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

test_loss = test_loss / len(test_loader)

print("Test Loss:", test_loss)

In [ ]:
accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted"
)

recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted"
)

f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

In [ ]:
all_labels_array = np.array(all_labels)
all_probabilities_array = np.array(all_probabilities)

# Convert labels to one-hot representation
true_one_hot = np.zeros_like(all_probabilities_array)

true_one_hot[
    np.arange(len(all_labels_array)),
    all_labels_array
] = 1

rmse = np.sqrt(
    mean_squared_error(
        true_one_hot,
        all_probabilities_array
    )
)

print(f"RMSE: {rmse:.4f}")

In [ ]:
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names
    )
)

In [ ]:
cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(10, 8))

plt.imshow(cm)

plt.colorbar()

plt.xticks(
    range(10),
    class_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(10),
    class_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("FashionMNIST Confusion Matrix")

plt.tight_layout()
plt.show()

In [ ]:
images, labels = next(iter(test_loader))

images = images.to(device)
labels = labels.to(device)

model.eval()

with torch.no_grad():

    outputs = model(images)

    _, predictions = torch.max(outputs, 1)

plt.figure(figsize=(12, 8))

for i in range(12):

    plt.subplot(3, 4, i + 1)

    image = images[i].cpu().squeeze()

    plt.imshow(image, cmap="gray")

    actual = class_names[labels[i].item()]
    predicted = class_names[predictions[i].item()]

    plt.title(
        f"Actual: {actual}\nPred: {predicted}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
torch.save(
    model.state_dict(),
    "fashion_mnist_cnn.pth"
)

print("Model saved successfully.")

In [ ]:
print("\n========== FINAL RESULTS ==========")

print(f"Test Loss  : {test_loss:.4f}")
print(f"Accuracy   : {accuracy * 100:.2f}%")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"F1 Score   : {f1:.4f}")
print(f"RMSE       : {rmse:.4f}")

print("===================================")

In [ ]:
#Observations
#The FashionMNIST dataset contains grayscale images of clothing items belonging to 10 different classes.
#The CNN automatically learns important image features using convolution and pooling layers.
#Training loss generally decreases as the number of epochs increases.
#Training accuracy generally increases with the number of epochs.
#The model can classify most FashionMNIST test images correctly.
#Some visually similar classes, such as Shirt, T-shirt/top, Coat and Pullover, may be confused with each other.
#Conclusion

#A Convolutional Neural Network was successfully implemented using PyTorch for FashionMNIST image classification. The images were preprocessed and passed through convolution, pooling and fully connected layers. The model was trained using the Adam optimizer and Cross Entropy Loss. Accuracy, precision, recall, F1-score and RMSE were calculated, and training graphs and a confusion matrix were used to visualize the results.